In [40]:
# 1. Prima celulă — importuri și citire dataset
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix
from sklearn import tree

# 1. Citim datasetul
data=pd.read_csv('diabetes.csv')

# 2. Verificam rapid datele
data.head()
data.shape
data.columns
data.describe()
data.isnull().sum()

# 3. Separăm coloanele de intrare de rezultatul pe care vrem să îl prezicem
x=data.drop(columns=['Outcome'])
y=data['Outcome']
x.head()
y.head()

# 4. Împărțim datele în date de antrenare și date de testare
x_train,x_test,y_train,y_test=train_test_split(
    x,y, test_size=0.2, random_state=42
)

print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

# 5. Creăm modelul
model=DecisionTreeClassifier(random_state=42)

# 6. Antrenăm modelul
model.fit(x_train, y_train)

# 7. Facem predicții pe datele de test
predictions=model.predict(x_test)
predictions

# 8. Evaluăm rezultatele
accuracy=accuracy_score(y_test, predictions)
recall=recall_score(y_test, predictions)
matrix=confusion_matrix(y_test, predictions)

print("Accurancy:",accuracy)
print("Recall:",recall)
print("Confusion matrix:")
print(matrix)

# 9. Testăm un pacient nou
new_patient=pd.DataFrame(
    [[
        2, 120, 70, 20, 80, 25.0, 0.5, 30
    ]],
    columns=x.columns
)

new_patient2=pd.DataFrame(
    [[
        6, 180, 90, 35, 160, 26.0, 0.8, 55
    ]],
    columns=x.columns
)
result=model.predict(new_patient2)

if result[0]==1:
    print("Modelul prezice risc de diabet.")
else:
    print("Modelul prezice fara diabet.")

print(" ")
print("Model imbunatatit.\n")
#Model imbunatatit

#1. Păstrăm codul vechi și facem o copie a datelor
data_clean=data.copy()

#2. Verificăm câte valori 0 suspecte avem
columns_to_fix=["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

print("Valori 0 inainte de curatare:")
for column in columns_to_fix:
    print(column,(data_clean[column]==0).sum())

#3. Înlocuim valorile 0 cu mediana
for column in columns_to_fix:
    median_value=data_clean[column].replace(0, np.nan).median()
    data_clean[column]=data_clean[column].replace(0, median_value)

print("Valori 0 dupa curatare:")
for column in columns_to_fix:
    print(column, (data_clean[column]==0).sum())

#4. Refacem X și y pe datele curate
x_clean=data_clean.drop(columns=["Outcome"])
y_clean=data_clean["Outcome"]

#5. Împărțim din nou datele
x_train_clean, x_test_clean, y_train_clean, y_test_clean= train_test_split(
    x_clean,y_clean, test_size=0.2, random_state=42
)
#80% din date sunt folosite pentru antrenare
#20% sunt păstrate pentru testare
#random_state=42 face ca rezultatul să fie același de fiecare dată când rulezi

#6. Antrenăm un model îmbunătățit
model_clean=DecisionTreeClassifier(
    random_state=42,
    class_weight="balanced",
    max_depth=4
)
#adăugăm class_weight="balanced", ca modelul să acorde mai multă atenție și clasei 1, adică pacienților cu diabet.
#max_depth=4 face arborele mai controlat, ca să nu învețe prea mult pe de rost.

model_clean.fit(x_train_clean, y_train_clean)

#7. Facem predicții și evaluăm
predictions_clean=model_clean.predict(x_test_clean)

accurancy_clean=accuracy_score(y_test_clean, predictions_clean)
recall_clean=recall_score(y_test_clean, predictions_clean)
matrix_clean=confusion_matrix(y_test_clean, predictions_clean)

print("Accurancy imbunatatit:",accurancy_clean)
print("Recall imbunatatit:",recall_clean)
print("Confusion matrix imbunatatit:")
print(matrix_clean)

#8. Variantă și mai orientată pe reducerea False Negative
#Dacă există măcar 40% probabilitate de diabet, mai bine marcăm ca risc.
#Asta poate reduce False Negative, dar poate crește False Positive. 
#În medical, uneori este acceptabil să ai mai multe alarme false dacă reduci cazurile ratate.

probabilities=model_clean.predict_proba(x_test_clean)[:,1]

predictions_threshold=[]

for probability in probabilities:
    if probability>=0.4:
        predictions_threshold.append(1)
    else:
        predictions_threshold.append(0)

accurancy_threshold=accuracy_score(y_test_clean, predictions_threshold)
recall_threshold=recall_score(y_test_clean, predictions_threshold)
matrix_threshold=confusion_matrix(y_test_clean, predictions_threshold)

print("Accurancy imbunatatitcu prag 0.4:",accurancy_threshold)
print("Recall imbunatatit cu prag 0.4:",recall_threshold)
print("Confusion matrix imbunatatit cu prag 0.4:")
print(matrix_threshold)

comparison = pd.DataFrame({
    "probability_diabetes": probabilities,
    "predict_default": predictions_clean,
    "predict_threshold_04": predictions_threshold
})

comparison.head(20)

(comparison["predict_default"] != comparison["predict_threshold_04"]).sum()

pd.Series(probabilities).value_counts().sort_index()

#9. Exportăm arborele pentru VS Code / Graphviz
tree.export_graphviz(
    model_clean,
    out_file="diabetes_tree_clean.dot",
    feature_names=x_clean.columns,
    class_names=["No Diabetes", "Diabetes"],
    label="all",
    rounded=True,
    filled=True
)

(614, 8)
(154, 8)
(614,)
(154,)
Accurancy: 0.7467532467532467
Recall: 0.7272727272727273
Confusion matrix:
[[75 24]
 [15 40]]
Modelul prezice risc de diabet.
 
Model imbunatatit.

Valori 0 inainte de curatare:
Glucose 5
BloodPressure 35
SkinThickness 227
Insulin 374
BMI 11
Valori 0 dupa curatare:
Glucose 0
BloodPressure 0
SkinThickness 0
Insulin 0
BMI 0
Accurancy imbunatatit: 0.7012987012987013
Recall imbunatatit: 0.8181818181818182
Confusion matrix imbunatatit:
[[63 36]
 [10 45]]
Accurancy imbunatatitcu prag 0.4: 0.7012987012987013
Recall imbunatatit cu prag 0.4: 0.8181818181818182
Confusion matrix imbunatatit cu prag 0.4:
[[63 36]
 [10 45]]
